# ONT Data Processing

In [ ]:
import pathlib
import tomllib

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import poreflow as pf  # poreFlow import
from poreflow.utils import show_text

%matplotlib widget

In this example, all the parameters are loaded from a single TOML file, to make it easy to keep track of the parameters used.

In [ ]:
with open("poreflow.toml", "rb") as f:
    config = tomllib.load(f)

input_config = config["input"]
file_name = input_config["file_name"]
folder_path = pathlib.Path(input_config["folder_path"])

with pf.File(folder_path / file_name) as f:
    device = f.device
    duration = f.duration

show_text(f"Trace from {device} device has duration: {duration:.2f} s")

## Inspect raw signals

Parameters

In [ ]:
ylim = (0, 300)
downsample_to = config["input"]["resample_to_freq"]

channel = 0  # Any of the 512 Channels in the ONT file
pct_range = (0, 1)  # only load data from % to %
bessel4_cutoff_plot = 10000

In [ ]:
times = (duration * pct_range[0], duration * pct_range[1])

with pf.File(folder_path / file_name) as f:
    print(
        "Plotting from {}s to {}s while downsampling from original sample rate of {} Hz to {} Hz".format(
            *times, f.sfreq, downsample_to
        )
    )

    raw = f.get_raw(channel, times)

raw = raw.downsample(downsample_to).apply_filter(bessel4_cutoff_plot)

fig, (ax_v, ax_i) = plt.subplots(
    2,
    1,
    figsize=(8, 4),
    dpi=150,
    sharex=True,
    gridspec_kw={"height_ratios": [1, 3]},
)

raw.plot("v", ax=ax_v)
raw.plot(ax=ax_i, stats=False)

ax_i.set_title(None)
ax_v.set_xlabel(None)

ax_i.set_ylim(ylim)

fig.tight_layout()

## Find events in one channel

In [ ]:
from poreflow.events import open_state, voltage_state

# Find events using the high-level API
events = raw.find_events(**config["event_finding"])
if events is None:
    raise ValueError("No events found")
min_duration = config["event_filtering"]["min_duration"]
events = events.filter_by_duration(min_duration, verbose=True)

# Recompute intermediate states for visualization
open_state_range = config["event_finding"]["open_state_range"]
voltage_range = config["event_finding"]["voltage_range"]
n_components = config["event_finding"]["n_components"]

component = open_state.find_open_state(raw, open_state_range, n_components)
raw["i_state"] = open_state.get_open_state_mask(raw, component)
raw["v_state"] = voltage_state.get_voltage_state_mask(raw, voltage_range)

print(f"Open state: {component[0]:.2f} pA, Open state fit: {raw.ios}")
print(events.head())

# Duration histogram
fig, ax = plt.subplots(figsize=(4, 3), dpi=150)
sns.histplot(events, x="duration", binwidth=1, ax=ax)
ax.set_ylabel("Count")
fig.tight_layout()

In [ ]:
from matplotlib.patches import Patch

xlim = (0, 650)

fig, (ax_v, ax_i) = plt.subplots(
    2,
    1,
    figsize=(6, 4),
    dpi=150,
    sharex=True,
    gridspec_kw={"height_ratios": [1, 3]},
)

# Voltage and current traces with state overlays
raw.plot("v", hue="v_state", ax=ax_v, stats=False)
raw.plot(ax=ax_i)

# Show event spans
for j, (start, end) in enumerate(zip(events["start_time"], events["end_time"])):
    if end < xlim[0] or start > xlim[1]:
        continue
    for ax in (ax_i, ax_v):
        ax.axvspan(start, end, color=".5", alpha=0.1, ec=None, zorder=-1)
    ax_i.text(
        (start + end) / 2,
        ylim[1],
        f"Event {j}",
        va="bottom",
        ha="center",
        color="0.5",
        size="small",
    )

ax_i.legend(
    handles=[Patch(facecolor="0.5", alpha=0.3, label="Detected events")],
    loc="upper left",
)
ax_i.set_title(None)
ax_v.set_xlabel(None)
ax_i.set_xlim(xlim)
ax_i.set_ylim(ylim)

fig.tight_layout()

## Show expected DNA signal pattern

In [ ]:
from poreflow.steps import predict

# seq from 5' to 3'
seq = config["boundary"]["template_DNA_5_3"]

steps_dna = predict.predict(seq)

fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
ax = sns.lineplot(steps_dna, x="step_idx", y="mean", drawstyle="steps-mid")
ax.set_ylabel("I/I$_{OS}$")
ax.set_title("Predicted DNA signal steps")
fig.tight_layout()

## Find events

In first 30 channels

In [ ]:
channel_list = (
    np.arange(0, 30) if device == pf.ONT else [0]
)  # List of channels for ONT device, for UTube, just first channel

In [ ]:
with pf.File(folder_path / file_name) as f:
    f.find_events(
        channels=channel_list,
        **config["event_finding"],
        verbose=config["env"]["verbose"],
        processes=2,
    )
    events = f.get_events()

## Find steps

In [ ]:
with pf.File(folder_path / file_name) as f:
    f.find_steps(**config["step_finding"], **config["env"])
    steps = f.get_steps()

In [ ]:
print(steps.head())

## Filter events

Adding more features

In [ ]:
from poreflow.events import selection

with pf.File(folder_path / file_name) as f:
    stats = selection.get_step_finding_stats(f)

print(stats.head())

Using features to filter steps 

In [ ]:
truth_table = selection.filter_from_config(stats, config["event_filtering"])

fig, ax = plt.subplots(figsize=(7, 2), dpi=150)
sns.heatmap(truth_table.T, cmap="Reds_r", ax=ax, vmin=0, vmax=1)

In [ ]:
mask = truth_table["all"]
print(
    f"Selecting {sum(mask)} out of {len(mask)} events ({sum(mask) / len(mask):.0%})"
)

In [ ]:
events_filtered = events.filter_by_mask(mask)
print(events_filtered.head())

## !!! Next step will remove filtered events on file !!!

In [ ]:
with pf.File(folder_path / file_name) as f:
    f.filter_events(mask)  # Also removes steps associated with removed events

## Viewing an event

In [ ]:
event_idx = 0

with pf.File(folder_path / file_name) as f:
    steps = f.get_steps(event_idx)
    event = f.get_event(event_idx)

fig, axs = plt.subplots(
    2,
    1,
    figsize=(8, 4),
    dpi=150,
    sharex=False,
    gridspec_kw={"height_ratios": [3, 1]},
)

event.apply_filter(1000).plot(x="rel", ax=axs[0])
axs[0].plot(steps["start_time"], steps["mean"], drawstyle="steps-post")

sns.lineplot(steps, x="step_idx", y="mean", drawstyle="steps-mid", ax=axs[1])

axs[0].set_ylim(25, 150)

fig.tight_layout()